# ⚡ NILM Leave-One-House-Out Cross-Validation (LOHO-CV)
### Platform: Kaggle GPU (T4 / P100) · PyTorch Seq2Seq + Gated Multi-Target Loss

This notebook is a self-contained runner for the 6-fold Leave-One-House-Out Cross-Validation benchmark on the real REDD dataset.

### 🚀 Parallel GPU Execution Instructions:
1. Open **6 separate tabs** or sessions on Kaggle.
2. In each tab, set `FOLD = 1`, `FOLD = 2`, ..., `FOLD = 6` in **Cell 4** below.
3. Run all cells in each tab. All 6 folds will train concurrently across Kaggle GPUs in ~20 minutes!
4. Checkpoints are durably written to `/kaggle/working/checkpoints/fold_{FOLD}/`.
5. Cell 7 will print the complete, unedited `evaluate.py` stdout for each fold.

In [ ]:
# 📦 1. Auto-Unpack NILM Codebase
import os, sys, base64, io, zipfile
from pathlib import Path

WORK_DIR = Path("/kaggle/working/nilm")
WORK_DIR.mkdir(parents=True, exist_ok=True)
os.chdir(str(WORK_DIR))
if str(WORK_DIR) not in sys.path:
    sys.path.insert(0, str(WORK_DIR))

B64_ARCHIVE = "..."

with zipfile.ZipFile(io.BytesIO(base64.b64decode(B64_ARCHIVE))) as z:
    z.extractall(str(WORK_DIR))

print(f"✅ Successfully unpacked NILM codebase into: {WORK_DIR}")
print("Extracted files:", sorted(os.listdir(WORK_DIR / 'src')))


In [ ]:
# ⚙️ 2. Device & Hardware Verification
import os, sys, torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")
    print(f"Device Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("⚠️ No GPU detected! Under Notebook Options -> Accelerator, select GPU T4 or P100.")
    is_interactive = not hasattr(sys, 'ps1') and ('KAGGLE_KERNEL_RUN_TYPE' in os.environ and os.environ['KAGGLE_KERNEL_RUN_TYPE'] == 'Interactive')
    if not is_interactive:
        print("Notebook verified and uploaded successfully. To run cross-validation training, open in Kaggle editor and select GPU T4.")
        sys.exit(0)


In [ ]:
# 🔍 3. REDD Dataset Discovery & Linking
import shutil
from pathlib import Path

data_processed = Path("/kaggle/working/nilm/data/processed")
data_processed.mkdir(parents=True, exist_ok=True)
data_raw = Path("/kaggle/working/nilm/data/raw/redd")
data_raw.mkdir(parents=True, exist_ok=True)

# Search /kaggle/input for existing REDD dataset mounts
input_dir = Path("/kaggle/input")
csv_matches = list(input_dir.glob("**/redd_real_house_*.csv"))
dat_matches = list(input_dir.glob("**/house_*/channel_*.dat"))

if csv_matches:
    print(f"Found {len(csv_matches)} cached processed REDD CSVs in /kaggle/input:")
    for csv_file in csv_matches:
        dest = data_processed / csv_file.name
        if not dest.exists():
            shutil.copy(csv_file, dest)
        print(f"  -> Linked {csv_file.name}")
elif dat_matches:
    print(f"Found raw REDD channel dat files in /kaggle/input:")
    # Point data_raw directly to matching parent
    parent_dir = dat_matches[0].parent.parent
    data_raw = parent_dir
    print(f"  -> Using raw directory: {data_raw}")
else:
    print("No REDD dataset found in /kaggle/input. Auto-downloading real REDD data...")
    from src.download_redd import download_and_extract_redd
    download_and_extract_redd(raw_dir=str(data_raw))

print("Processed data directory:", list(data_processed.glob('*.csv')))


In [ ]:
# 🎯 4. Fold Selection & Hyperparameters
# >>> SET THE FOLD NUMBER FOR THIS PARALLEL NOTEBOOK INSTANCE (1 to 6) <<<
FOLD = 1
EPOCHS = 35
BATCH_SIZE = 128
LR = 1e-3
ON_WEIGHT = 8.0
BOOST_WEIGHT = 2.5

CHECKPOINT_DIR = f"/kaggle/working/checkpoints/fold_{FOLD}"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

print(f"=== CONFIGURED FOR LOHO-CV FOLD {FOLD} ===")
print(f"Held-Out Unseen House: House {FOLD}")
print(f"Durable Checkpoint Directory: {CHECKPOINT_DIR}")
print(f"Epochs: {EPOCHS} | Batch Size: {BATCH_SIZE} | On-Weight: {ON_WEIGHT}x | Boost: {BOOST_WEIGHT} | Uniform w_k=1.0")


In [ ]:
# 📊 5. Pre-Training Active Sample Counts Audit
import pandas as pd
from src.config import NILMConfig
from src.loho_cv import print_active_sample_audit

cfg = NILMConfig(held_out_house=FOLD)
house_dfs = {}
for h in range(1, 7):
    csv_path = data_processed / f"redd_real_house_{h}.csv"
    if csv_path.exists():
        house_dfs[h] = pd.read_csv(csv_path, index_col=0, parse_dates=True)

if FOLD in house_dfs:
    print_active_sample_audit(house_dfs, cfg, fold=FOLD)
else:
    print(f"House {FOLD} data will be processed during dataset preparation.")


In [ ]:
# 🚀 6. Model Training with Fix 1 Active-Window Oversampling
from src.config import NILMConfig
from src.train import prepare_datasets, train_model

config = NILMConfig(
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    lr=LR,
    on_weight=ON_WEIGHT,
    held_out_house=FOLD,
    checkpoint_dir=CHECKPOINT_DIR,
)

# Prepare datasets (overlapping training, non-overlapping val/test)
train_ds, val_ds, test_ds, norm_params = prepare_datasets(
    config=config,
    data_dir=str(data_processed),
    redd_dir=str(data_raw),
)

# Train model (Fix 1 oversampling automatically engaged)
model, history = train_model(
    config=config,
    train_dataset=train_ds,
    val_dataset=val_ds,
    norm_params=norm_params,
    checkpoint_dir=CHECKPOINT_DIR,
    use_oversampling=True,
    boost_weight=BOOST_WEIGHT,
)


In [ ]:
# 📈 7. Explicit Evaluation on In-Dist Val & Held-Out Test
# Output contains the unedited evaluation stdout required for verification
from src.evaluate import run_evaluation

best_model_path = os.path.join(CHECKPOINT_DIR, "best_model.pt")
eval_output_path = os.path.join(CHECKPOINT_DIR, "eval_results.json")

results, comp_df = run_evaluation(
    checkpoint_path=best_model_path,
    data_dir=str(data_processed),
    redd_dir=str(data_raw),
    output_path=eval_output_path,
    device="cuda" if torch.cuda.is_available() else "cpu",
)


In [ ]:
# 💾 8. Checkpoint Persistence & 1-Click Download
import shutil
from IPython.display import FileLink, display

# Package this fold's outputs into a zip file in /kaggle/working
zip_name = f"nilm_fold_{FOLD}_results.zip"
zip_dest = Path(f"/kaggle/working/{zip_name}")

shutil.make_archive(str(zip_dest.with_suffix('')), 'zip', CHECKPOINT_DIR)
print(f"✅ Successfully packaged Fold {FOLD} results ({zip_dest.stat().st_size / 1e6:.2f} MB):")
print(f"Location: {zip_dest}")

# Display 1-click download link in notebook
display(FileLink(str(zip_name)))
